In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
from config import RAW_DATA_PATH, PROCESSED_DATA_PATH
from preprocess import (
    process_time_columns,
    extract_eco_code,
    extract_opening_name,
    extract_opening_family,
    count_moves,
    avg_time_per_move,
    build_player_features,
    add_rating_change,
    remove_unnecessary_columns,
)

## 1. Load Data

In [2]:
df = pd.read_csv(RAW_DATA_PATH, index_col=0)
print(f'Shape: {df.shape}')
df.head()

Shape: (1286, 26)


,uuid,start_time,end_time,time_control,time_class,rated,rules,initial_setup,eco,match,...,white_username,white_rating,white_result,white_uuid,white_id,black_username,black_rating,black_result,black_uuid,black_id
url,,,,,,,,,,,,,,,,,,,,,
https://www.chess.com/game/live/129536542471,3bb0deed-c8ce-11ef-82f6-6cfe544c0428,NaN,1735797692,600,rapid,True,chess,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,https://www.chess.com/openings/Mieses-Opening,NaN,...,badaayasher,1374,win,de2c233a-854e-11ea-8432-ff2c02fe170d,https://api.chess.com/pub/player/badaayasher,thien3703,1093,abandoned,d5c59aba-c837-11ef-9435-7b3d26a453d4,https://api.chess.com/pub/player/thien3703
https://www.chess.com/game/live/122748087628,1e5aadc4-e76f-11ef-92f0-c442db01000f,NaN,1739165505,600,rapid,True,chess,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,https://www.chess.com/openings/Queens-Pawn-Ope...,NaN,...,thien3703,1217,win,d5c59aba-c837-11ef-9435-7b3d26a453d4,https://api.chess.com/pub/player/thien3703,oldnewschool,1078,resigned,83bf4358-da39-11ed-af9a-d9e04a0af27d,https://api.chess.com/pub/player/oldnewschool
https://www.chess.com/game/live/132904194763,58b82041-e770-11ef-b220-6cfe544c0428,NaN,1739166291,600,rapid,True,chess,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,https://www.chess.com/openings/French-Defense-...,NaN,...,Alfredo_DBO,1202,resigned,d7a6c88e-805f-11ee-a558-4904d035ae6d,https://api.chess.com/pub/player/alfredo_dbo,thien3703,1308,win,d5c59aba-c837-11ef-9435-7b3d26a453d4,https://api.chess.com/pub/player/thien3703
https://www.chess.com/game/live/122870154412,b67b1102-ea95-11ef-99e1-b584ca01000f,NaN,1739512233,600,rapid,True,chess,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,https://www.chess.com/openings/Sicilian-Defens...,NaN,...,thien3703,1383,win,d5c59aba-c837-11ef-9435-7b3d26a453d4,https://api.chess.com/pub/player/thien3703,ajkmr2241,1306,resigned,69179e6e-7f85-11ee-b61e-21edc9db7b76,https://api.chess.com/pub/player/ajkmr2241
https://www.chess.com/game/live/122870389378,a2176cb7-ea97-11ef-99e1-b584ca01000f,NaN,1739512807,600,rapid,True,chess,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,https://www.chess.com/openings/Queens-Gambit-D...,NaN,...,ajkmr2241,1316,win,69179e6e-7f85-11ee-b61e-21edc9db7b76,https://api.chess.com/pub/player/ajkmr2241,thien3703,1308,checkmated,d5c59aba-c837-11ef-9435-7b3d26a453d4,https://api.chess.com/pub/player/thien3703


In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 1286 entries, https://www.chess.com/game/live/129536542471 to https://www.chess.com/game/live/170403592806
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   uuid            1286 non-null   str    
 1   start_time      0 non-null      float64
 2   end_time        1286 non-null   int64  
 3   time_control    1286 non-null   str    
 4   time_class      1286 non-null   str    
 5   rated           1286 non-null   bool   
 6   rules           1286 non-null   str    
 7   initial_setup   1286 non-null   str    
 8   eco             1286 non-null   str    
 9   match           0 non-null      float64
 10  tournament      0 non-null      float64
 11  fen             1286 non-null   str    
 12  tcn             1286 non-null   str    
 13  pgn             1286 non-null   str    
 14  white_accuracy  300 non-null    float64
 15  black_accuracy  300 non-null    float64
 16  white_usern

## 2. Preprocessing

In [4]:
# Xử lí cả hai cột thời gian cùng lúc
df = process_time_columns(df)

# Opening features từ PGN
df['opening_type']   = df['pgn'].apply(extract_eco_code)
df['opening']        = df['pgn'].apply(extract_opening_name)
df['opening_family'] = df['opening'].apply(extract_opening_family)

# Game features từ PGN
df['num_moves']         = df['pgn'].apply(count_moves)
df['avg_time_per_move'] = df['pgn'].apply(avg_time_per_move)

df = build_player_features(df)
df = add_rating_change(df)
df = remove_unnecessary_columns(df)

# Đưa start_time lên đầu
df = df[['start_time'] + [c for c in df.columns if c != 'start_time']]

print(f'Shape: {df.shape}')
df.head()

Shape: (1286, 15)


,start_time,end_time,time_control,rated,opening_type,opening,opening_family,num_moves,avg_time_per_move,player_color,player_rating,opponent_rating,player_result,result_method,rating_change
0,2025-01-02 05:55:56,2025-01-02 06:01:32,600,True,A00,Mieses Opening,Mieses Opening,63,5.226667,black,1093,1374,Loss,Abandon,0
1,2025-02-10 05:23:10,2025-02-10 05:31:45,600,True,D00,Queens Pawn Opening 1...d5 2.e3,Queens Pawn Opening,39,12.436842,white,1217,1078,Win,Resign,124
2,2025-02-10 05:31:58,2025-02-10 05:44:51,600,True,C00,French Defense Knight Variation,French Defense,70,17.250000,black,1308,1202,Win,Resign,91
3,2025-02-14 05:37:00,2025-02-14 05:50:33,600,True,B20,Sicilian Defense 2.d3,Sicilian Defense,75,10.664865,white,1383,1306,Win,Resign,75
4,2025-02-14 05:50:45,2025-02-14 06:00:07,600,True,D31,Queens Gambit Declined Queens Knight Variation...,Queens Gambit,45,14.976190,black,1308,1316,Loss,Checkmate,-75


In [5]:
df.to_csv(PROCESSED_DATA_PATH)